<a href="https://colab.research.google.com/github/franklinzhou-ncsu/new_repo_554/blob/main/ST_554_Homework_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Subject: ST 554 - Homework 8

Name: Franklin Zhou

Date: 3/29/2026

For this homework you will continue your notebook from homework 7 and add a few models to it! Create a copy of your notebook and add to it according to the below instructions.

# Goal
The purpose of this homework is to practice fitting tree based models using `scikit-learn`

# Data

We will use the same dataset from the UCI Machine Learning Repository we used in homework 7.
- Rather than try to predict quality, let's make our target variable for fitting regression type models alcohol.
- For fitting classification type models we'll use the type of wine as the response variable.

# To Do:

Add to your previous document. See below.

In [1]:
# Read data sets and merge them (Homework 7)
import pandas as pd
import numpy as np

# read in the datasets
df_red_wine = pd.read_csv('winequality-red.csv', sep=';')
df_white_wine = pd.read_csv('winequality-white.csv', sep=';')

# create a new variable: red wine = 0 and white wine = 1
df_red_wine['type'] = 0
df_white_wine['type'] = 1

# combine the two datasets
df_wine = pd.concat([df_red_wine, df_white_wine])

In [2]:
# Split data and add polynomial items (Homework 7)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures

# Split the data into training and test sets with stratification
X_train, X_test, y_train, y_test = train_test_split(
    df_wine.drop("alcohol", axis=1),
    df_wine["alcohol"],
    test_size=0.2,
    random_state=41,
    stratify = df_wine['type'])

# add polynomial terms
poly2 = PolynomialFeatures(degree=2, include_bias=False)

X_train_poly = poly2.fit_transform(X_train)
X_test_poly = poly2.fit_transform(X_test)

In [3]:
# Build 4 best models (Homework 7)
from math import sqrt
from sklearn.linear_model import LinearRegression, LogisticRegression, LogisticRegressionCV, LassoCV, Lasso, Ridge, RidgeCV, ElasticNetCV, ElasticNet
from sklearn.model_selection import cross_validate

# The best MLR model
mlr_best = LinearRegression().fit(X_train_poly, y_train)

# LASSO CV Model
lasso_mod = LassoCV(cv=5, random_state=0).fit(X_train, y_train) # Use all predictors
lasso_best = Lasso(lasso_mod.alpha_).fit(X_train,y_train)

# Ridge CV Model
ridge_mod = RidgeCV(cv=5, alphas=np.logspace(-2, 2, 100)).fit(X_train, y_train)
ridge_best = Ridge(ridge_mod.alpha_).fit(X_train,y_train)

# Elastic Net CV Model
en_mod = ElasticNetCV(cv=5, random_state=0, l1_ratio = np.linspace(0.001, 0.999, 100), n_alphas = 50).fit(X_train, y_train)
en_best = ElasticNet(alpha = en_mod.alpha_, l1_ratio = en_mod.l1_ratio_).fit(X_train,y_train)


## Regression Task (`alcohol` as Response)

## Train Models



### Regression Tree


- Fit a regression tree
    - Use at least five predictors
    - Use CV to select the tuning parameters (`max_depth` and `min_samples_leaf`)

In [4]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV

# Tune the model
rtree_tune = GridSearchCV(DecisionTreeRegressor(),
 {'max_depth': range(2,15),'min_samples_leaf': range(2,20)},
                          cv = 5,
                          scoring = "neg_mean_squared_error").fit(X_train, y_train)


In [5]:
print(rtree_tune.best_estimator_)

DecisionTreeRegressor(max_depth=13, min_samples_leaf=6)


The best tuning parameters are: `max_depth = 13` and `min_samples_leaf = 6`

In [6]:
# Train the tree model with best estimators
rtree_cv = rtree_tune.best_estimator_.fit(X_train, y_train)

### Random Forest

- Fit a random forest model
    - Use at least five predictors
    - Use CV to select the tuning parameter(s) (`max_features`, feel free to include others if you'd like)

In [7]:
from sklearn.ensemble import RandomForestRegressor

# Setup parameters
parameters = {"max_features": range(1,5), "max_depth": [3, 4, 5, 10, 15], 'min_samples_leaf':[3, 10, 50]}

rf_tune = GridSearchCV(RandomForestRegressor(n_estimators = 500),
                       parameters,
                       cv = 5,
                       scoring = "neg_mean_squared_error").fit(
                           X_train[['quality', 'density', 'volatile acidity', 'chlorides', 'type', 'citric acid']],
                           y_train)


In [8]:
print(rf_tune.best_estimator_)

RandomForestRegressor(max_depth=15, max_features=4, min_samples_leaf=3,
                      n_estimators=500)


With independent variables `quality`, `density`, `volatile acidity`, `chlorides`, `type`, and `citric acid`, The best tuning parameters are: `max_depth = 15`, `max_features = 4`, and `min_samples_leaf = 3`

In [9]:
# Train the RF model with best estimators
rf_cv = rf_tune.best_estimator_.fit(X_train, y_train)

## Test Models

- Using your two selected models, compare their performance on the test set **including a comparison to your models from homework 7!**

    - Do so using RMSE as your model metric
    - Do so using MAE as your model metric

In [10]:
# Test models on test dataset
from sklearn.metrics import mean_squared_error, mean_absolute_error

# make prediction using each model
mlr_pred = mlr_best.predict(X_test_poly)
lasso_pred = lasso_best.predict(X_test)
ridge_pred = ridge_best.predict(X_test)
en_pred = en_best.predict(X_test)
rtree_pred = rtree_cv.predict(X_test)
rf_pred = rf_cv.predict(X_test)

# create a dict for 6 models
models_pred = {
    "Best_MLR": mlr_pred,
    "LASSO": lasso_pred,
    "Ridge": ridge_pred,
    "Elastic_Net": en_pred,
    "Reg_Tree": rtree_pred,
    "Random_Forest": rf_pred
}

results = []

# calculate the RMSE and MAE for each model
for name, pred in models_pred.items():
    RMSE = np.sqrt(mean_squared_error(y_test, pred))
    MAE = mean_absolute_error(y_test, pred)
    results.append([name, RMSE, MAE])

# Output as a table
pd.DataFrame(results, columns=["Model", "RMSE", "MAE"])

,Model,RMSE,MAE
0,Best_MLR,0.414295,0.311780
1,LASSO,0.964009,0.771769
2,Ridge,0.668847,0.530374
3,Elastic_Net,0.964011,0.771773
4,Reg_Tree,0.541143,0.387498
5,Random_Forest,0.457389,0.334617


From the output we can see that the Best_MLR model has the lowest RMSE and MAE value.

## Classification Task (Wine Type as Response)

- Repeat the training and testing done previously but use classification trees and random forests with classification trees.
- Use log-loss or negative log-loss as your metric for choosing models during the training process
- During the testing portion, compare all of (including homework 7) your models on both log-loss and accuracy

### Retrieve code from homework 7

In [11]:
# From Homework 7
# Split the data into training and test sets with stratification
X_train, X_test, y_train, y_test = train_test_split(
    df_wine.drop("type", axis=1),
    df_wine["type"],
    test_size=0.2,
    random_state=41,
    stratify = df_wine['type'])

means = X_train.apply(np.mean, axis = 0)
stds = X_train.apply(np.std, axis = 0)

# quick function to standardize based off of a supplied mean and std
def my_std_fun (x, means, stds):
    return(x-means)/stds

# standardize the X_train data set
for x in X_train.columns:
    X_train[x] = my_std_fun(X_train[x], means[x], stds[x])

# standardize the X_test data set
for x in X_test.columns:
    X_test[x] = my_std_fun(X_test[x], means[x], stds[x])

# add interaction terms
poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
X_train_inter = poly.fit_transform(X_train)

# add polynomial terms
poly2 = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly2.fit_transform(X_train)

# select some variables
X_train_select = X_train[['chlorides', 'residual sugar', 'quality', 'density']]

In [12]:
# Build 4 best models (Homework 7)

# best logistic regression model
log_reg_best_cv = LogisticRegression(
    solver = "newton-cg",
    penalty = None,
    max_iter = 5000).fit(X_train, y_train)

# best lasso model
log_lasso_cv = LogisticRegressionCV(
    solver = "liblinear",
    penalty = "l1",
    Cs = 250,
    scoring = "neg_log_loss", # use negative log-loss as metric
    random_state = 10).fit(X_train, y_train)

log_lasso_best_cv = LogisticRegression(
    solver = "liblinear",
    penalty = "l1",
    C = log_lasso_cv.C_[0],
    random_state = 10).fit(X_train, y_train)

# best Ridge model
log_ridge_cv = LogisticRegressionCV(
    solver = "newton-cg",
    penalty = "l2",
    Cs = 250,
    scoring = "neg_log_loss", # use negative log-loss as metric
    random_state = 10).fit(X_train, y_train)

log_ridge_best_cv = LogisticRegression(
    solver = "newton-cg",
    penalty = "l2",
    C = log_ridge_cv.C_[0], # the best C value from CV
    random_state = 5).fit(X_train, y_train)

# best Elasticnet model
log_en_cv = LogisticRegressionCV(
    solver = "saga", # The only solver that works with elasticnet
    penalty = "elasticnet",
    l1_ratios = np.linspace(0.01, 0.99, 20),
    Cs = 10,
    scoring = "neg_log_loss", # use negative log-loss as metric
    max_iter = 5000,
    random_state = 10).fit(X_train, y_train)

log_en_best_cv = LogisticRegression(
    solver = "saga",
    penalty = "elasticnet",
    l1_ratio = log_en_cv.l1_ratio_[0], # the best l1_ratio value from CV
    C = log_en_cv.C_[0], # the best C value from CV
    max_iter = 5000,
    random_state = 5).fit(X_train, y_train)

## Train Models


### Classification Tree

In [17]:
from sklearn.tree import DecisionTreeClassifier
parameters = {'max_depth': range(2,15), 'min_samples_leaf':[3, 5, 10, 15, 50]}
cltree_model = GridSearchCV(DecisionTreeClassifier(), parameters, cv = 5, scoring='neg_log_loss').fit(X_train, y_train)
print(cltree_model.best_estimator_)

DecisionTreeClassifier(max_depth=4, min_samples_leaf=5)


So, the best parameters for classification tree model are: `max_depth = 4` and `min_samples_leaf = 5`.

In [18]:
best_cltree = cltree_model.best_estimator_.fit(X_train, y_train)

### Random Forest


In [19]:
from sklearn.ensemble import RandomForestClassifier
parameters = {"max_features": range(1,5), "max_depth": [3, 4, 5, 10, 15], 'min_samples_leaf':[3, 10, 50]}
rf_model = GridSearchCV(RandomForestClassifier(), parameters, cv = 5, scoring='neg_log_loss').fit(X_train, y_train)
print(rf_model.best_estimator_)

RandomForestClassifier(max_depth=10, max_features=4, min_samples_leaf=3)


The best parameters for random forest model are: `max_depth = 10`, `max_features = 4` and `min_samples_leaf = 3`.


In [20]:
best_rftree = rf_model.best_estimator_.fit(X_train, y_train)

## Test Models

In [21]:
# make prediction using each model
log_reg_best_cv_pred = log_reg_best_cv.predict(X_test)
log_lasso_best_cv_pred = log_lasso_best_cv.predict(X_test)
log_ridge_best_cv_pred = log_ridge_best_cv.predict(X_test)
log_en_best_cv_pred = log_en_best_cv.predict(X_test)
cltree_best_cv_pred = best_cltree.predict(X_test)
rftree_best_cv_pred = best_rftree.predict(X_test)

# make predicted probability using each model
log_reg_best_cv_pred_prob = log_reg_best_cv.predict_proba(X_test)
log_lasso_best_cv_pred_prob = log_lasso_best_cv.predict_proba(X_test)
log_ridge_best_cv_pred_prob = log_ridge_best_cv.predict_proba(X_test)
log_en_best_cv_pred_prob = log_en_best_cv.predict_proba(X_test)
cltree_best_cv_pred_prob = best_cltree.predict_proba(X_test)
rftree_best_cv_pred_prob = best_rftree.predict_proba(X_test)

In [22]:
from sklearn.metrics import accuracy_score, log_loss

results = []

results.append([
    "Logistic Regression",
    log_loss(y_test, log_reg_best_cv_pred_prob),
    accuracy_score(y_test, log_reg_best_cv_pred)
])

results.append([
    "LASSO",
    log_loss(y_test, log_lasso_best_cv_pred_prob),
    accuracy_score(y_test, log_lasso_best_cv_pred)
])

results.append([
    "Ridge",
    log_loss(y_test, log_ridge_best_cv_pred_prob),
    accuracy_score(y_test, log_ridge_best_cv_pred)
])

results.append([
    "Elastic Net",
    log_loss(y_test, log_en_best_cv_pred_prob),
    accuracy_score(y_test, log_en_best_cv_pred)
])

results.append([
    "Classification Tree",
    log_loss(y_test, cltree_best_cv_pred_prob),
    accuracy_score(y_test, cltree_best_cv_pred)
])

results.append([
    "Random Forest Classification Tree",
    log_loss(y_test, rftree_best_cv_pred_prob),
    accuracy_score(y_test, rftree_best_cv_pred)
])

pd.DataFrame(results, columns=["Model", "Log-Loss", "Accuracy"])


,Model,Log-Loss,Accuracy
0,Logistic Regression,0.043299,0.992308
1,LASSO,0.041404,0.992308
2,Ridge,0.044359,0.991538
3,Elastic Net,0.044197,0.991538
4,Classification Tree,0.080119,0.979231
5,Random Forest Classification Tree,0.036703,0.994615


From the output we can see that the Random Forest classification tree model has the highest accuracy and the lowest log-loss.

# Submit

- Make sure all cells are run in your notebook
- Submit the link to your github repo